In [0]:

#lendo arquivo da tabela bronze para transformações.
df = spark.table("`cafe_sales_2026`.bronze.dirty_cafe_sales")
display(df)

In [0]:
# . Após analise dos formatos, vamos converter colunas numéricas usando try_cast (valores inválidos viram null)/por que usar try_cast? é mais usado em produção e protege o processo.
from pyspark.sql.functions import expr

tipos = {
    "Quantity": "INT",
    "Price_Per_Unit": "DOUBLE",
    "Total_Spent": "DOUBLE"
    
}

df_clean = df

for coluna, tipo in tipos.items():
    df_clean = df_clean.withColumn(
        coluna,
        expr(f"try_cast({coluna} AS {tipo})")
    )

In [0]:
#Data é mais sensível/ melhor opção e fazer separadamento a sua transformação.
from pyspark.sql.functions import expr

df_clean = df_clean.withColumn(
    "Transaction_Date",
    expr("try_to_date(Transaction_Date)")
)

In [0]:
#Corrigindo coluna Total_Spent usando cálculo Quantity * price_per_unit)
from pyspark.sql.functions import col, when

df_clean = df_clean.withColumn(
    "Total_Spent",
    when(
        col("Total_Spent").isNull(),
        col("Quantity") * col("Price_Per_Unit")
    )
    .otherwise(col("Total_Spent"))
)

In [0]:
df_clean.printSchema()

In [0]:
#revisando tabela para a poxima transfomação;
display(df_clean)

In [0]:
from pyspark.sql.functions import col, when

df_clean = (
    df_clean
    .withColumn(
        "Quantity",
        when(
            col("Quantity").isNull() &
            col("Price_Per_Unit").isNotNull() &
            col("Total_Spent").isNotNull(),
            col("Total_Spent") / col("Price_Per_Unit")
        )
        .otherwise(col("Quantity"))
        .cast("int")
    )
    .withColumn(
        "Price_Per_Unit",
        when(
            col("Price_Per_Unit").isNull() &
            col("Quantity").isNotNull() &
            col("Total_Spent").isNotNull(),
            col("Total_Spent") / col("Quantity")
        )
        .otherwise(col("Price_Per_Unit"))
        .cast("double")
    )
    .withColumn(
        "Total_Spent",
        when(
            col("Total_Spent").isNull() &
            col("Quantity").isNotNull() &
            col("Price_Per_Unit").isNotNull(),
            col("Quantity") * col("Price_Per_Unit")
        )
        .otherwise(col("Total_Spent"))
        .cast("double")
    )
)

In [0]:
display(df_clean)

In [0]:
df_clean.printSchema()

In [0]:
from pyspark.sql.functions import col, when, trim

# Valores que devem virar null
valores_invalidos = ["null", "NaN", "None", "ERROR", "UNKNOWN", "-"]

for coluna in df_clean.columns:
    tipo_coluna = dict(df_clean.dtypes)[coluna]
    
    if tipo_coluna == "string":
        # Só aplica trim() em colunas string
        df_clean = df_clean.withColumn(
            coluna,
            when(
                (col(coluna).isNull()) |
                (trim(col(coluna)) == "") |
                (trim(col(coluna)).isin(valores_invalidos)),
                None
            ).otherwise(col(coluna))
        )
    else:
        # Colunas numéricas: só verifica null (não dá pra ter "ERROR" numa coluna já convertida pra number)
        df_clean = df_clean.withColumn(
            coluna,
            when(col(coluna).isNull(), None).otherwise(col(coluna))
        )

display(df_clean)

In [0]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cafe_sales_2026.silver.clean_cafe_sales")